# 🔁 End-to-End RAG Evaluation

## Putting It All Together

In this notebook we run a **mini evaluation** of a complete RAG pipeline:

```
Test questions → Retrieve → Generate → Evaluate both stages
```

You'll see exactly how to build an evaluation harness for your own RAG system.

In [1]:
import re
import math
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer('all-MiniLM-L6-v2')
print("Ready!")

Ready!


## 1. The Knowledge Base

In [2]:
# Small knowledge base — each doc has an ID
knowledge_base = [
    {"id": "doc_0", "text": "Dropout randomly deactivates neurons during training to prevent overfitting."},
    {"id": "doc_1", "text": "Dropout is applied only during training, not at inference time."},
    {"id": "doc_2", "text": "Typical dropout rates are 0.2 to 0.5."},
    {"id": "doc_3", "text": "Batch normalization normalizes layer inputs to speed up and stabilize training."},
    {"id": "doc_4", "text": "Batch normalization helps reduce internal covariate shift in deep networks."},
    {"id": "doc_5", "text": "Transfer learning reuses weights from a pretrained model on a new related task."},
    {"id": "doc_6", "text": "Transfer learning is especially useful when training data is scarce."},
    {"id": "doc_7", "text": "Gradient descent minimizes loss by updating weights in small steps."},
    {"id": "doc_8", "text": "The learning rate controls how large each gradient descent step is."},
]

# Embed all docs
kb_texts = [d["text"] for d in knowledge_base]
kb_embeddings = model.encode(kb_texts)

print(f"Knowledge base: {len(knowledge_base)} documents")

Knowledge base: 9 documents


## 2. The Test Set

A test set has: questions, correct answers, and which docs *should* be retrieved.

In [3]:
test_set = [
    {
        "question": "What is dropout and when is it applied?",
        "reference_answer": "Dropout randomly deactivates neurons during training to prevent overfitting. It is only applied during training, not at inference.",
        "relevant_doc_ids": {"doc_0", "doc_1"},
    },
    {
        "question": "How does batch normalization help training?",
        "reference_answer": "Batch normalization normalizes layer inputs, which speeds up and stabilizes training by reducing internal covariate shift.",
        "relevant_doc_ids": {"doc_3", "doc_4"},
    },
    {
        "question": "What is transfer learning useful for?",
        "reference_answer": "Transfer learning reuses pretrained model weights for a new task and is especially useful when training data is scarce.",
        "relevant_doc_ids": {"doc_5", "doc_6"},
    },
]

print(f"Test set: {len(test_set)} questions")

Test set: 3 questions


## 3. The RAG Pipeline

In [4]:
# Retrieval
def retrieve(question, top_k=3):
    q_emb = model.encode(question)
    scores = cosine_similarity([q_emb], kb_embeddings)[0]
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [
        {"id": knowledge_base[i]["id"], "text": knowledge_base[i]["text"], "score": float(scores[i])}
        for i in top_indices
    ]

In [5]:
# Generation (simulated LLM)
# Replace this with a real LLM call in production

def generate(question, retrieved_docs):
    """
    Simulates an LLM response by joining retrieved doc texts.
    Real version would call:
        client.messages.create(model=..., messages=[...], system=...)
    """
    # Simulate: the LLM roughly summarizes the retrieved docs
    simulated_answers = {
        "What is dropout and when is it applied?":
            "Dropout randomly deactivates neurons during training to prevent overfitting. It is only used during training, not at inference time.",
        "How does batch normalization help training?":
            "Batch normalization normalizes inputs to each layer, helping speed up training and reduce instability.",
        "What is transfer learning useful for?":
            "Transfer learning is useful when you have limited training data. It reuses weights from a model trained on a related task.",
    }
    return simulated_answers.get(
        question,
        " ".join(d["text"] for d in retrieved_docs[:2])  # fallback: concat top 2 docs
    )

## 4. Run Evaluation

In [6]:
# Metric helpers (from previous notebooks)

def recall_at_k(retrieved_ids, relevant_ids, k):
    if not relevant_ids: return 0.0
    hits = sum(1 for doc_id in retrieved_ids[:k] if doc_id in relevant_ids)
    return hits / len(relevant_ids)

def normalize(text):
    return re.sub(r'\s+', ' ', re.sub(r'[^\w\s]', '', text.lower())).strip()

def token_f1(pred, ref):
    p_tokens = set(normalize(pred).split())
    r_tokens = set(normalize(ref).split())
    if not p_tokens or not r_tokens: return 0.0
    common = p_tokens & r_tokens
    prec = len(common) / len(p_tokens)
    rec  = len(common) / len(r_tokens)
    return 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0

def semantic_sim(pred, ref):
    return float(cosine_similarity([model.encode(pred)], [model.encode(ref)])[0][0])

print("Metric helpers ready.")

Metric helpers ready.


In [7]:
# Run the full evaluation loop

results = []

for sample in test_set:
    question = sample["question"]

    # Step 1: Retrieve
    retrieved = retrieve(question, top_k=3)
    retrieved_ids = [d["id"] for d in retrieved]

    # Step 2: Generate
    generated = generate(question, retrieved)

    # Step 3: Measure retrieval
    r_at_3 = recall_at_k(retrieved_ids, sample["relevant_doc_ids"], 3)

    # Step 4: Measure generation
    f1   = token_f1(generated, sample["reference_answer"])
    sem  = semantic_sim(generated, sample["reference_answer"])

    results.append({
        "question": question,
        "retrieved_ids": retrieved_ids,
        "generated": generated,
        "recall@3": r_at_3,
        "token_f1": f1,
        "semantic_sim": sem,
    })

# Print results
print("End-to-End Evaluation Results\n")
print(f"{'Question':<42} {'Recall@3':>9} {'TokenF1':>8} {'SemSim':>8}")
print("-" * 72)

for r in results:
    print(f"{r['question']:<42} {r['recall@3']:>9.2f} {r['token_f1']:>8.3f} {r['semantic_sim']:>8.3f}")

print("\nAverages:")
for metric in ["recall@3", "token_f1", "semantic_sim"]:
    avg = sum(r[metric] for r in results) / len(results)
    print(f"  {metric:<15}: {avg:.3f}")

End-to-End Evaluation Results

Question                                    Recall@3  TokenF1   SemSim
------------------------------------------------------------------------
What is dropout and when is it applied?         1.00    0.909    0.989
How does batch normalization help training?      1.00    0.533    0.925
What is transfer learning useful for?           1.00    0.632    0.827

Averages:
  recall@3       : 1.000
  token_f1       : 0.691
  semantic_sim   : 0.914


## 5. Diagnose What's Failing

In [8]:
# Use the scores to understand WHERE the pipeline is weak

print("Diagnosis\n")

for r in results:
    retrieval_ok  = r["recall@3"] >= 0.8
    generation_ok = r["semantic_sim"] >= 0.8

    if retrieval_ok and generation_ok:
        status = "✅ Both good"
    elif retrieval_ok and not generation_ok:
        status = "⚠️  Retrieval OK, Generation weak — improve prompt"
    elif not retrieval_ok and generation_ok:
        status = "⚠️  Generation OK but retrieval weak — LLM used training memory!"
    else:
        status = "❌ Both weak — start with fixing retrieval"

    print(f"  {status}")
    print(f"  Q: {r['question']}")
    print(f"  Retrieved: {r['retrieved_ids']}")
    print(f"  Answer: {r['generated'][:80]}...")
    print()

Diagnosis

  ✅ Both good
  Q: What is dropout and when is it applied?
  Retrieved: ['doc_1', 'doc_2', 'doc_0']
  Answer: Dropout randomly deactivates neurons during training to prevent overfitting. It ...

  ✅ Both good
  Q: How does batch normalization help training?
  Retrieved: ['doc_3', 'doc_4', 'doc_7']
  Answer: Batch normalization normalizes inputs to each layer, helping speed up training a...

  ✅ Both good
  Q: What is transfer learning useful for?
  Retrieved: ['doc_6', 'doc_5', 'doc_8']
  Answer: Transfer learning is useful when you have limited training data. It reuses weigh...



## Key Takeaways 🎯

### The Evaluation Loop:

```
1. Build a test set (questions + ground truth docs + reference answers)
2. Run your RAG pipeline on each question
3. Measure retrieval: Recall@K
4. Measure generation: Token F1, Semantic Similarity, Faithfulness
5. Diagnose: is retrieval or generation the bottleneck?
6. Fix the weakest stage first
7. Re-evaluate
```

### What to Improve First:

| Problem | Fix |
|---|---|
| Low Recall@K | Better chunking, better embedding model, hybrid search |
| Low Faithfulness | Stronger grounding prompt, fewer retrieved docs |
| Low Semantic Sim | Better prompt, more context, query decomposition |
| Both low | Start with retrieval — generation can't fix bad retrieval |

---
**Congratulations — you've completed the core RAG pipeline!**

```
✅ Section 1: Embeddings
✅ Section 2: Retrieval
✅ Section 3: Search Techniques
✅ Section 4: Generation
✅ Section 6: Evaluation
```